# Hyperparameters

*updated 2/2/25*

#### Hyperparameters are external configuration settings that control the training process of a machine learning model but are not learned from the data. For example, the degree of the polynomial is a hyperparameter.  It determines the complexity of the polynomial function. A higher degree allows for more flexibility but increases the risk of overfitting. Since hyperparameters are not directly learned from the data, they need to be manually set or optimized using techniques like Grid search.
#### The code below demonstrates how to use K-fold cross-validation and grid search for hyperparameter tuning.

In [ ]:
# First, import all the packages we'll need.
import matplotlib.pyplot as plt

import numpy as np
# reduce display precision on numpy arrays
np.set_printoptions(precision=1)

import pandas as pd
# reduce display precision on pandas dataframes
pd.set_option('display.precision', 1)

from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict, KFold, GridSearchCV
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
import joblib

In [ ]:
# Next, download the data.

#example data for ML task1 polynomial regression for intial run through is
#url='https://raw.githubusercontent.com/yanwu2001/DOE-ML-Public/main/PhysicalCatapult/ML_Catapult_ABC_y.csv'
#updated raw data url for retraining the model is
#url='https://raw.githubusercontent.com/yanwu2001/DOE-ML-Public/main/PhysicalCatapult/Catapult_ABCDEFGy_raw.csv'
# update the URL below as needed when retraining the model
url='https://raw.githubusercontent.com/yanwu2001/DOE-ML-Public/main/PhysicalCatapult/Catapult_ABCDEFGy_raw.csv'
df=pd.read_csv(url)

# Use the ".head()" and ".info()" and ".describe()" to learn more about the dataset.
print("This is the header:\n", df.head())
print("\nThis is the info:\n")
df.info() # this command doesn't need to be 'print'ed
print("\nThis is the description:\n",df.describe())

This is the header:
       ID  Combo#  ball_mass  firing_height  band_length  band_height  \
0  DGJH1       9       60.0           11.0         52.5         28.2   
1  DGJH2       9       94.0           15.9         55.0         28.2   
2  DGJH3       9       94.0            5.6         55.0         28.2   
3  DGJH4       9       23.0           15.9         55.0         28.2   
4  DGJH5       9       23.0            5.6         55.0         28.2   

   collar_height  collar_distance  firing_position  launch_distance short_long  
0            6.5             54.8             18.5            289.4       long  
1            6.5             54.8             18.5            253.0       long  
2            6.5             54.8             18.5            342.0       long  
3            6.5             54.8             18.5            251.0       long  
4            6.5             54.8             18.5            446.5       long  

This is the info:

<class 'pandas.core.frame.DataFrame'>
Ra

## Data pre-processing
####There are clearly columns we don't want to use in our fit, such as "Combo#" and "ID".

In [ ]:
# SO, we must CLEAN the data. Separate the predictors and the labels:

# Define data for y_data by selecting the 'launch_distance' column
y_data=df['launch_distance'].copy()

# Get rid of non relevant data
X_data=df.drop(['ID','Combo#','launch_distance','short_long'],axis=1)

# Impute missing values with the mean of the that attrubute.
#  Other available strategies: "median", "most_frequent"
imputer = SimpleImputer(strategy='mean')
X_data = imputer.fit_transform(X_data)

# Randomly, assign 80% of the dataset as the training set. Put the remaining 20% into the test set.
#  NOTE the 2025 modification - not using an explicit cross validation set, yet - see below.
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=1)

# Randomly split the 20% subset above into two: one half for cross validation and the other for the test set
#X_cv, X_test, y_cv, y_test = train_test_split(X_, y_, test_size=0.50, random_state=1)
# Printing headers {.head()} shows the column headers plus the first few rows of data.
print("Headers for training set:\n",X_train.head())

# Printing the description {.describe()} shows the # of data points, mean, standard deviation, median (50%), and more.
#  This view is helpful if we want to identify outliers (mean ≠ median) and also
#    to identify if the quantities have vastly different numerical values (and so need scaling to a -1 --> +1 scale).
print("\nDescription of training set:\n",X_train.describe())
print("\nDescription of test set:\n",X_test.describe())
print("\nDescription of training targets:\n",y_train.describe())

Headers for training set:
      ball_mass  firing_height  band_length  band_height  collar_height  \
62        94.0            9.0         39.0         28.5           13.0   
127       23.0           18.3         57.3         14.6            1.9   
111       60.0           18.3         57.3         13.4            1.9   
288       68.5            7.4         47.3         34.0            9.0   
108       94.0           18.3         57.3          9.3            1.9   

     collar_distance  firing_position  
62              49.5             18.6  
127             49.2             16.7  
111             49.2             16.7  
288             59.5             21.0  
108             49.2             10.8  

Description of training set:
        ball_mass  firing_height  band_length  band_height  collar_height  \
count      243.0          243.0        243.0        243.0          243.0   
mean        55.0           14.3         47.1         26.5           10.5   
std         32.5            4

## Train and Evaluate Model using K-fold Cross-validation
The previous code assigns 80% of the dataset as the training set and keeps the remaining 20% for the test set. We want to avoid using the test set until we are confident in our model, which means we should utilize part of the training set for training and another part for model validation.

Cross-validation is a technique used to evaluate a machine learning model’s performance by dividing the dataset into multiple subsets. During this process, the model is trained on some subsets while tested on others. This approach helps estimate the model's ability to generalize to new data, thereby reducing the risk of overfitting or underfitting.

The most common method of cross-validation is k-fold cross-validation, which involves partitioning the dataset into k subsets. In this method, each subset is used as a test set while the model is trained on the remaining k-1 subsets.


 The following code randomly splits the training set into 10 non-overlapping subsets called folds. Then, it trains and evaluates the Linear Regression model 10 times, selecting a different fold for evaluation each time and using the other 9 folds for training. The result is an array containing the 10 evaluation scores. It’s important to note that in scikit-learn, the cross-validation scThe following code randomly divides the training set into 10 non-overlapping subsets, known as folds. It then trains and evaluates the Linear Regression model 10 times, using a different fold for evaluation each time while utilizing the other 9 folds for training. The result is an array that contains the evaluation scores from each of the 10 runs. It's important to note that in scikit-learn, the cross-validation score is displayed as a negative value, which represents the opposite of the Root Mean Square Error (RMSE). Thus, a higher score (or smaller absolute value) indicates better model performance.ore is presented as a negative value, which is the opposite of the Root Mean Square Error (RMSE). Therefore, a higher score (or smaller absolute value) indicates better performance.

In [ ]:
train_utility=cross_val_score(LinearRegression(), X_train, y_train, scoring='neg_root_mean_squared_error', cv=10)
print("Cross_validation score:\n", train_utility)
kfold_rmses=pd.DataFrame(-train_utility)# Turn the negative value to positive for RMSEs
print("\nDescription of KFold results in terms of RMSE statistics :\n",kfold_rmses.describe())


Cross_validation score:
 [-37.7 -29.  -64.  -44.9 -41.7 -46.4 -39.1 -32.3 -36.5 -50.1]

Description of KFold results in terms of RMSE statistics :
           0
count  10.0
mean   42.2
std    10.0
min    29.0
25%    36.8
50%    40.4
75%    46.1
max    64.0


## Fine-tune the model using Grid Search and K-folds cross-validation
####We will use k-folds cross-validation to determine the "best" polynomial order for our model. This is another way of saying that we will use cross validation to find the best *hyperparameter* for our linear fit. Here, the "hyperparameter" value determines whether we fit to a 1st-order model, or quadratic, or cubic, or... (etc.)
#### The polynomial order is considered a hyperparameter because it is not something that is adjusted when the model fits parameters to reduce the loss - i.e., it is something that "we" provide & it is unchanged during the fitting process.

####  The following code searches for the best polynomial order for the Linear Regression model. For each polynomial order, Kfold cross-validation produces a mean loss score with a standard deviation. This process is repeated for each of the polynomial orders that we investigate. This is done using the GridSearchCV() object.
#### At the end, the GridSearchCV provides the scores, and the "winner:" the polynomial order with the lowest mean loss score (highest score since the score is negative value).



In [ ]:
# Create a pipeline. This ensures that the polynomial features created by PolynomialFeatures (i.e. A^2, AB^2, etc.) are scaled to a -1 --> +1
#  scale before performing the linear regression.
pipe = Pipeline([
    ('poly', PolynomialFeatures()), # Creates n-th order features (i.e. if 2, creates A^2, B^2, ... and AB, BC, ...)
    ('scaler', StandardScaler()),  # Scaling step. Other scalers exist.
    ('reg', LinearRegression())   # The type of regression for our model fit.
])

# Define the parameter grid
param_grid = {
    'poly__degree': [1, 2, 3, 4, 5]  # Test different polynomial degrees. "poly__" indicates a parameter of PolynomialFeatures
                                        # will be varied; "degree" indicates that it's the 'degree' parameter that we'll vary.
}

# Create the cross-variance folds (KFolds).
#  This creates 10 folds, i.e. 90% of the training set will be used at any one time for the fit;
#   the remaining 10% will be saved for the cross-validation check. [NOTE that the X_test data is UNTOUCHED during this process!]
folds=KFold(n_splits=10, shuffle=True) # be sure to shuffle!

# Create the GridSearchCV object. It uses the pipeline, parameter grid, and K-folds defined above.
#  Here, we also select the scoring method that will be used to determine the "best" model.
grid = GridSearchCV(pipe, param_grid, cv=folds, scoring='neg_root_mean_squared_error')

# Fit the grid search
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)

# Create a DataFrame from the cv_results_ attribute
cv_results = pd.DataFrame(grid.cv_results_)

# Print the mean test scores for each degree
print("Mean test scores for each degree:")
print(cv_results[['param_poly__degree', 'mean_test_score', 'std_test_score']])

# NOTE that when I re-run this, I sometimes get poly__degree = 3 as the best  fit!

{'poly__degree': 2}
Mean test scores for each degree:
   param_poly__degree  mean_test_score  std_test_score
0                   1            -42.9             5.6
1                   2            -26.8             6.4
2                   3           -252.9           531.3
3                   4            -43.2            59.0
4                   5            -42.8            58.7


 In metrology, uncertainty quantifies the confidence in a measurement's accuracy due to inherent variability and limitations in measurement tools. Similarly, cross-validation provides an estimate of a model’s uncertainty by evaluating its performance across different subsets of data.  By averaging the k-fold cross-validation results and analyzing their spread (e.g., standard deviation of accuracy scores), we get a measure similar to uncertainty in measurement, helping assess how reliable the model's predictions are on unseen data.Considering the results shown above, can you confidently identify which degree of the polynomial provides the best model?

## Select the best model
#### We can select the best model (first order, second order.  ) based on the grid search results. The following codes print out the selection.

In [ ]:
# We can select the best model.
best_model = grid.best_estimator_

# Print the parameters of the best model
print("ALL parameters of the best model:")
print(best_model.get_params())

# Print the best parameters that were VARIED during the grid search:
print("Best varied parameters:", grid.best_params_)

# Print the coefficients. SIDE NOTE, this seems like a pain & perhaps a diversion from other more important concepts. Perhaps not with students.
# Access the linear regression step in the pipeline
linear_regression = best_model.named_steps['reg']
# Print the coefficients.
print("Best coefficients:", linear_regression.coef_)


ALL parameters of the best model:
{'memory': None, 'steps': [('poly', PolynomialFeatures()), ('scaler', StandardScaler()), ('reg', LinearRegression())], 'transform_input': None, 'verbose': False, 'poly': PolynomialFeatures(), 'scaler': StandardScaler(), 'reg': LinearRegression(), 'poly__degree': 2, 'poly__include_bias': True, 'poly__interaction_only': False, 'poly__order': 'C', 'scaler__copy': True, 'scaler__with_mean': True, 'scaler__with_std': True, 'reg__copy_X': True, 'reg__fit_intercept': True, 'reg__n_jobs': None, 'reg__positive': False}
Best varied parameters: {'poly__degree': 2}
Best coefficients: [-2.7e-10 -5.2e+01 -2.0e+00  5.8e+02  1.1e+02  6.2e+02 -6.4e+02  1.5e+02
  3.8e+01  2.0e+01 -4.2e+00  5.1e+00  6.9e+00 -9.2e+00 -1.9e+01  8.5e-02
 -1.3e+02  3.4e+02 -2.6e+01 -7.0e+01 -2.3e+02 -4.6e+02 -3.3e+02 -3.5e+02
  3.8e+02 -9.5e+00 -3.8e+02  8.5e+01  5.9e+02  6.7e+01  1.4e+01 -3.6e+02
 -1.9e+01  6.6e+01  1.8e+01 -1.9e+01]


## Deploy the model, test it using test dataset

In [ ]:
# We can USE the best model.
# Use the best model to make predictions on the test data
y_pred = best_model.predict(X_test)

# Print predictions
print("Predictions on test data:", y_pred)

# Calculate RMSE on the test data
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE on test data:", rmse)

# Calculate NMSE on the test data
#nmse = -mean_squared_error(y_test, y_pred)
#print("Negative MSE on test data:",nmse)

Predictions on test data: [174.9 154.8 181.6 145.1 153.  189.6 329.8 177.8 153.  176.   59.2 332.5
 119.8 206.5 186.7 182.5 293.8 144.8 293.3 144.8 376.9 182.5 197.1 227.2
  74.9 195.5 152.4 188.3 293.3 256.3 176.  250.7 165.3 184.1 267.5 125.
 190.9 158.7 332.5  76.5 173.9 206.5 227.2 161.4 189.6 175.8 216.3 440.8
 495.  117.7 188.3 157.5 293.3 101.7 235.5 101.7 293.3 282.1 181.6 157.5
  99.9]
RMSE on test data: 24.479147101573393


Does the RMSE on the test data confirm our selection of hyperparameter (degree of polynomial)?  Give your reasoning.

For example, the RMSE on the test data (24.5) meets our expectations, as the K-fold validation shows our best model has a mean RMSE of 25.9 with a standard deviation of 4.9.